# Лабораторная работа 1. BPE и Unigram токенизаторы с нуля (студенческая версия)

**Курс:** Методы обработки естественного языка · **Блок 1:** языковое моделирование и токенизация

**Цель.** Понять токенизацию как задачу сжатия и оценить, как выбор токенизатора влияет на последующую модель.

**Что нужно сдать:**
1. Реализацию byte-level BPE и Unigram LM токенизаторов без сторонних библиотек (кроме `re`, `collections`, `math`).
2. Тесты корректности: \(\mathrm{decode}(\mathrm{encode}(x)) = x\) на всём корпусе и на произвольных байтах.
3. Сравнение с `sentencepiece` (совпадение сегментаций, скорость).
4. Графики «размер словаря — степень сжатия» для русского и английского корпусов и ответы на вопросы.

**Теория, которую нужно уметь воспроизвести на защите.** BPE — жадный алгоритм сжатия: на каждом шаге в словарь добавляется пара символов с наибольшей частотой. Unigram LM задаёт вероятностную модель сегментации
\[
p(x) = \sum_{s \in S(x)} \prod_{i} p(s_i),
\]
где \(S(x)\) — множество разбиений строки на подслова. Параметры оцениваются EM-алгоритмом: E-шаг вычисляет ожидаемые числа вхождений подслов через прямой и обратный проход (forward-backward), M-шаг нормирует их. Кодирование — поиск самой вероятной сегментации алгоритмом Витерби.

In [ ]:
# ==== Общая настройка (одинакова во всех работах) ====
import os, math, time, json, random, pathlib, urllib.request
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

FAST_DEV_RUN = True          # True — сокращённый прогон (CPU, минуты); False — полный эксперимент
SEED = 42
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
ARTIFACTS = pathlib.Path("artifacts"); ARTIFACTS.mkdir(exist_ok=True)
DATA = pathlib.Path("data"); DATA.mkdir(exist_ok=True)

def set_seed(seed=SEED):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
set_seed()

def download(url, name):
    path = DATA / name
    if not path.exists():
        urllib.request.urlretrieve(url, path)
    return path

def load_shakespeare():
    url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
    return download(url, "shakespeare.txt").read_text(encoding="utf-8")

def load_russian():
    """Русский корпус: «Война и мир» (Л. Н. Толстой), т. 1, из открытого репозитория.
    При недоступности сети используется встроенный фрагмент."""
    urls = ["https://raw.githubusercontent.com/mmcdermott/nlp-war-and-peace/master/data/war_and_peace_ru.txt",
            "https://raw.githubusercontent.com/nevmenandr/word2vec-russian-novels/master/books_before/WarAndPeace.txt"]
    for i, u in enumerate(urls):
        try:
            t = download(u, f"ru_{i}.txt").read_text(encoding="utf-8", errors="ignore")
            if len(t) > 100_000: return t
        except Exception:
            pass
    base = ("Все счастливые семьи похожи друг на друга, каждая несчастливая семья несчастлива по-своему. "
            "Всё смешалось в доме Облонских. Жена узнала, что муж был в связи с бывшею в их доме француженкою-гувернанткой, "
            "и объявила мужу, что не может жить с ним в одном доме. ")
    return base * 400

print(f"torch {torch.__version__}, device={DEVICE}, FAST_DEV_RUN={FAST_DEV_RUN}")

## 0. Данные

Используем два корпуса: англоязычный (Tiny Shakespeare) и русскоязычный («Война и мир»). В режиме `FAST_DEV_RUN` корпуса обрезаются.

In [ ]:
en_text = load_shakespeare()
ru_text = load_russian()
LIMIT = 200_000 if FAST_DEV_RUN else 2_000_000
en_text, ru_text = en_text[:LIMIT], ru_text[:LIMIT]
print(len(en_text), len(ru_text))
print(ru_text[5000:5300])

## 1. Byte-level BPE

Алгоритм обучения:
1. Разбить текст на «предтокены» регулярным выражением (слова с ведущим пробелом, числа, пунктуация), чтобы слияния не пересекали границы слов.
2. Представить каждый предтокен как последовательность байтов UTF-8 (идентификаторы 0–255).
3. Повторять `vocab_size - 256` раз: найти самую частую соседнюю пару, создать новый токен, заменить все вхождения пары.

Наивный пересчёт всех пар на каждом шаге стоит \(O(\text{корпус})\), поэтому храните счётчики пар и индекс «пара → предтокены, где она встречается», обновляя их только для затронутых предтокенов.

In [ ]:
import re
from collections import Counter, defaultdict

PRETOKEN_RE = re.compile(r" ?[^\W\d_]+| ?\d{1,3}| ?[^\s\w]+| ?_+|\s+(?!\S)|\s+", re.UNICODE)

def pretokenize(text):
    return PRETOKEN_RE.findall(text)

assert "".join(pretokenize("Hello, мир! 12345  x")) == "Hello, мир! 12345  x"
print(pretokenize("Hello, мир! 12345  x"))

In [ ]:
def merge_word(word, pair, new_id):
    """Заменяет все вхождения pair в кортеже word на new_id (слева направо, без перекрытий)."""
    merged, i = [], 0
    while i < len(word):
        if i + 1 < len(word) and (word[i], word[i + 1]) == pair:
            merged.append(new_id)
            i += 2
        else:
            merged.append(word[i])
            i += 1
    return tuple(merged)

assert merge_word((1, 2, 3, 1, 2), (1, 2), 9) == (9, 3, 9)
assert merge_word((1, 1, 1), (1, 1), 7) == (7, 1)

In [ ]:
class ByteBPE:
    def __init__(self):
        self.merges = {}                       # (a, b) -> new_id, порядок вставки = ранг
        self.vocab = {i: bytes([i]) for i in range(256)}
        self._cache = {}

    def train(self, text, vocab_size, verbose=False):
        assert vocab_size >= 256
        words = Counter(tuple(t.encode("utf-8")) for t in pretokenize(text))
        word_list = list(words.keys()); counts = [words[w] for w in word_list]
        self.merges = {}
        self.vocab = {i: bytes([i]) for i in range(256)}
        pair_counts, pair_words = Counter(), defaultdict(set)
        # Одинаковый предтокен хранится один раз; counts[i] учитывает все его вхождения.
        for i, word in enumerate(word_list):
            for pair, count in Counter(zip(word, word[1:])).items():
                pair_counts[pair] += count * counts[i]
                pair_words[pair].add(i)
        while len(self.vocab) < vocab_size and pair_counts:
            # Частое слияние жадно сокращает последовательность; глобальный
            # оптимум по всем возможным порядкам слияний не ищется.
            pair = min(pair_counts, key=lambda p: (-pair_counts[p], p))
            new_id = len(self.vocab)
            self.merges[pair] = new_id
            self.vocab[new_id] = self.vocab[pair[0]] + self.vocab[pair[1]]
            # Обновляем пары только в предтокенах, где произошло слияние.
            changed = set()
            for i in sorted(pair_words[pair]):
                old = word_list[i]
                new = merge_word(old, pair, new_id)
                before = Counter(zip(old, old[1:]))
                after = Counter(zip(new, new[1:]))
                for p in before.keys() | after.keys():
                    pair_counts[p] += (after[p] - before[p]) * counts[i]
                    if after[p]:
                        pair_words[p].add(i)
                    else:
                        pair_words[p].discard(i)
                    changed.add(p)
                word_list[i] = new
            for p in changed:
                if pair_counts[p] <= 0:
                    del pair_counts[p]
            if verbose and new_id % 500 == 0:
                print('Размер словаря:', new_id + 1)
        self._cache = {}
        return self

    def _encode_chunk(self, chunk):
        if chunk in self._cache: return self._cache[chunk]
        ids = tuple(chunk.encode("utf-8"))
        # При кодировании выбираем самое раннее из доступных обученных слияний.
        # Частоты пар в новом тексте уже не пересчитываются.
        while len(ids) > 1:
            pair = min(zip(ids, ids[1:]), key=lambda p: self.merges.get(p, math.inf))
            if pair not in self.merges:
                break
            ids = merge_word(ids, pair, self.merges[pair])
        self._cache[chunk] = list(ids)
        return list(ids)

    def encode(self, text):
        out = []
        for ch in pretokenize(text): out.extend(self._encode_chunk(ch))
        return out

    def decode(self, ids):
        return b''.join(self.vocab[i] for i in ids).decode('utf-8')

t0 = time.time()
bpe_en = ByteBPE().train(en_text, 1000)
print(f"обучение: {time.time() - t0:.1f} c, токенов в словаре: {len(bpe_en.vocab)}")
print([bpe_en.vocab[i] for i in list(bpe_en.vocab)[256:276]])

### Тесты корректности

Проверьте обратимость на всём корпусе, на тексте на другом языке, на эмодзи и на случайных байтах, декодированных как UTF-8 с заменой.

In [ ]:
def check_roundtrip(tok, text):
    return tok.decode(tok.encode(text)) == text

assert check_roundtrip(bpe_en, en_text)
assert check_roundtrip(bpe_en, ru_text[:20000])
assert check_roundtrip(bpe_en, "Ёжик 🦔 и 数学 — test\t\n  ")
rnd = bytes(np.random.randint(0, 256, 5000).tolist()).decode("utf-8", errors="replace")
assert check_roundtrip(bpe_en, rnd)
print("Все тесты обратимости пройдены")

## 2. Unigram LM токенизатор

Работаем на уровне символов Unicode, пробел заменяем на `▁` и присоединяем к началу слова (как в SentencePiece).

1. **Начальный словарь:** все символы корпуса + самые частые подстроки длины 2…`max_len` (в несколько раз больше целевого размера).
2. **E-шаг:** для каждого слова прямой проход \(\alpha_j = \log\sum_{i<j} \exp(\alpha_i + \log p(x_{i:j}))\), обратный проход \(\beta\); ожидаемое число вхождений подслова \(x_{i:j}\) — \(\exp(\alpha_i + \log p(x_{i:j}) + \beta_j - \log Z)\), умноженное на частоту слова.
3. **M-шаг:** \(p(s) \propto \mathbb{E}[\text{count}(s)]\).
4. **Прунинг:** удалить долю подслов с наименьшей вероятностью (одиночные символы не удалять), повторять до целевого размера словаря.

Это упрощённая версия: SentencePiece удаляет подслова по приросту правдоподобия корпуса при их удалении, а не по вероятности.

In [ ]:
def logsumexp2(a, b):
    if a == -math.inf: return b
    if b == -math.inf: return a
    m = max(a, b); return m + math.log(math.exp(a - m) + math.exp(b - m))

def words_of(text):
    return Counter("▁" + w for w in text.split())

class UnigramTokenizer:
    def __init__(self, max_len=8):
        self.max_len = max_len; self.logp = {}

    def _forward_backward(self, w):
        """Возвращает (logZ, словарь ожидаемых вхождений подслов) для одного слова."""
        n = len(w)
        edges = [[] for _ in range(n)]
        for i in range(n):
            for j in range(i + 1, min(n, i + self.max_len) + 1):
                piece = w[i:j]
                if piece in self.logp:
                    edges[i].append((j, piece, self.logp[piece]))
        # alpha[j] суммирует вероятности всех путей до позиции j.
        # Считаем в логарифмах, чтобы произведения малых вероятностей не обращались в ноль.
        alpha = [-math.inf] * (n + 1)
        alpha[0] = 0.0
        for i in range(n):
            for j, piece, lp in edges[i]:
                alpha[j] = logsumexp2(alpha[j], alpha[i] + lp)
        log_z = alpha[n]
        if log_z == -math.inf:
            raise ValueError('Словарь не покрывает символы слова')
        # beta[i] так же учитывает все продолжения от позиции i до конца слова.
        beta = [-math.inf] * (n + 1)
        beta[n] = 0.0
        for i in range(n - 1, -1, -1):
            for j, piece, lp in edges[i]:
                beta[i] = logsumexp2(beta[i], lp + beta[j])
        # Деление на Z даёт вероятность каждого перехода при условии данного слова.
        # Складываем эти вероятности и получаем ожидаемое число вхождений подслова.
        expected = Counter()
        for i in range(n):
            for j, piece, lp in edges[i]:
                expected[piece] += math.exp(alpha[i] + lp + beta[j] - log_z)
        # В отличие от BPE, обучение учитывает все сегментации, а не один путь.
        # Сэмплирование путей могло бы использоваться для subword regularization.
        return log_z, expected

    def train(self, text, vocab_size, init_factor=4, prune_frac=0.2, em_iters=2, verbose=True):
        words = words_of(text)
        chars = Counter(); subs = Counter()
        for w, c in words.items():
            for ch in w: chars[ch] += c
            for i in range(len(w)):
                for j in range(i + 2, min(len(w), i + self.max_len) + 1):
                    subs[w[i:j]] += c
        init = dict(chars); init.update(dict(subs.most_common(init_factor * vocab_size)))
        total = sum(init.values()); self.logp = {s: math.log(c / total) for s, c in init.items()}
        self.chars = set(chars)
        while True:
            for _ in range(em_iters):
                # E-шаг учитывает частоту слова в корпусе, M-шаг нормирует ожидаемые частоты.
                expected, ll = Counter(), 0.0
                for word, count in words.items():
                    log_z, word_expected = self._forward_backward(word)
                    ll += count * log_z
                    for piece, value in word_expected.items():
                        expected[piece] += count * value
                total = sum(max(expected[piece], 1e-200) for piece in self.logp)
                self.logp = {piece: math.log(max(expected[piece], 1e-200) / total)
                             for piece in self.logp}
            if verbose: print(f"|V|={len(self.logp):6d}  log-likelihood={ll:.4e}")
            if len(self.logp) <= vocab_size: break
            # Одиночные символы сохраняем: иначе часть слов корпуса нельзя будет разбить.
            removable = sorted((s for s in self.logp if len(s) > 1),
                               key=lambda s: (self.logp[s], s))
            n_remove = min(len(removable), len(self.logp) - vocab_size,
                           max(1, int(len(self.logp) * prune_frac)))
            if n_remove == 0:
                raise ValueError('Целевой словарь меньше обязательного алфавита')
            for piece in removable[:n_remove]:
                del self.logp[piece]
            # После удаления подслов сумма оставшихся вероятностей снова должна равняться 1.
            log_total = math.log(sum(math.exp(lp) for lp in self.logp.values()))
            self.logp = {piece: lp - log_total for piece, lp in self.logp.items()}
        return self

    def segment(self, w):
        """Витерби: самая вероятная сегментация слова."""
        n = len(w)
        # best[j] хранит оценку лучшего пути, previous[j] нужен для восстановления разбиения.
        best, previous = [-math.inf] * (n + 1), [None] * (n + 1)
        best[0] = 0.0
        unknown_lp = min(self.logp.values()) - 10.0
        for i in range(n):
            for j in range(i + 1, min(n, i + self.max_len) + 1):
                piece = w[i:j]
                lp = self.logp.get(piece)
                if lp is None:
                    if j != i + 1:
                        continue
                    # Невстреченный символ оставляем отдельным фрагментом с низкой оценкой.
                    lp = unknown_lp
                score = best[i] + lp
                if score > best[j]:
                    best[j], previous[j] = score, (i, piece)
        pieces, j = [], n
        while j:
            i, piece = previous[j]
            pieces.append(piece)
            j = i
        return pieces[::-1]

    def encode(self, text):
        # Сохраняем точную обратимость: пробельные символы кодируются отдельными «токенами»
        out = []
        for part in re.split(r"(\s+)", text):
            if not part: continue
            if part.isspace():
                if part == " ": continue           # одиночный пробел кодируется префиксом ▁ следующего слова
                out.append(part)
            else:
                out.extend(self.segment("▁" + part))
        return out

    def decode(self, pieces):
        s, prev_ws = "", True
        for p in pieces:
            if p.startswith("▁"):
                s += ("" if prev_ws else " ") + p[1:]; prev_ws = False
            else:
                s += p; prev_ws = p.isspace()
        return s

t0 = time.time()
uni_en = UnigramTokenizer().train(en_text[: (60_000 if FAST_DEV_RUN else 500_000)], 1000)
print(f"обучение: {time.time() - t0:.1f} c")
print(uni_en.encode("To be, or not to be: that is the question."))

**Замечание об обратимости.** Наш Unigram-токенизатор работает с токенами-строками, а не с байтами. Обратимость гарантирована для текстов из одиночных пробелов между словами; проверьте её на корпусе, нормализованном так, как это делает SentencePiece (подряд идущие пробелы сжаты, переводы строк сохранены).

In [ ]:
def normalize_spaces(t):
    t = re.sub(r"[ \t]+", " ", t)
    return re.sub(r" *\n *", "\n", t).strip()

sample = normalize_spaces(en_text[:20000])
assert uni_en.decode(uni_en.encode(sample)) == sample
print("Unigram: обратимость на нормализованном корпусе выполнена")

## 3. Сравнение с SentencePiece

Обучите `sentencepiece` в режимах `bpe` и `unigram` с тем же размером словаря и сравните:
- долю слов, сегментированных одинаково с вашей реализацией;
- среднее число токенов на слово;
- время обучения и кодирования.

Полного совпадения ожидать не нужно: SentencePiece использует другой прунинг, нормализацию и обработку редких символов.

In [ ]:
try:
    import sentencepiece as spm
except ImportError:
    os.system("pip -q install sentencepiece"); import sentencepiece as spm

corpus_file = DATA / "en_corpus.txt"; corpus_file.write_text(en_text, encoding="utf-8")
spm_models = {}
for mtype in ("bpe", "unigram"):
    t0 = time.time()
    spm.SentencePieceTrainer.train(input=str(corpus_file), model_prefix=str(ARTIFACTS / f"spm_{mtype}"),
                                   vocab_size=1000, model_type=mtype, character_coverage=1.0,
                                   max_sentence_length=100000, minloglevel=2)
    spm_models[mtype] = spm.SentencePieceProcessor(model_file=str(ARTIFACTS / f"spm_{mtype}.model"))
    print(mtype, f"обучение {time.time() - t0:.1f} c")

In [ ]:
def agreement(seg_a, seg_b, words):
    same = sum(seg_a(w) == seg_b(w) for w in words)
    return same / len(words)

test_words = [w for w, _ in words_of(en_text[150_000:]).most_common(2000)]
ours_uni = lambda w: uni_en.segment(w)
sp_uni = lambda w: [p for p in spm_models["unigram"].encode(w[1:], out_type=str)]
ours_bpe = lambda w: [bpe_en.vocab[i].decode("utf-8", "replace") for i in bpe_en.encode(" " + w[1:])]
sp_bpe = lambda w: [p.replace("▁", " ") for p in spm_models["bpe"].encode(w[1:], out_type=str)]

import pandas as pd

# Для сравнения обучаем все четыре модели на одинаковом en_text.
train_seconds = {}
t0 = time.perf_counter()
bpe_en = ByteBPE().train(en_text, 1000)
train_seconds['BPE'] = time.perf_counter() - t0
t0 = time.perf_counter()
uni_en = UnigramTokenizer().train(en_text, 1000, verbose=False)
train_seconds['Unigram'] = time.perf_counter() - t0
for mtype in ('bpe', 'unigram'):
    t0 = time.perf_counter()
    spm.SentencePieceTrainer.train(input=str(corpus_file),
        model_prefix=str(ARTIFACTS / f'spm_{mtype}'), vocab_size=1000,
        model_type=mtype, character_coverage=1.0,
        max_sentence_length=100000, minloglevel=2)
    spm_models[mtype] = spm.SentencePieceProcessor(
        model_file=str(ARTIFACTS / f'spm_{mtype}.model'))
    train_seconds['SP ' + mtype] = time.perf_counter() - t0

comparison = []
for name, segment, reference, encode in [
    ('BPE', ours_bpe, sp_bpe, bpe_en.encode),
    ('SP bpe', sp_bpe, sp_bpe, spm_models['bpe'].encode),
    ('Unigram', ours_uni, sp_uni, uni_en.encode),
    ('SP unigram', sp_uni, sp_uni, spm_models['unigram'].encode),
]:
    times = []
    for _ in range(3):
        if name == 'BPE': bpe_en._cache.clear()
        t0 = time.perf_counter()
        encode(en_text)
        times.append(time.perf_counter() - t0)
    comparison.append({'токенизатор': name,
        'совпадение с SP': agreement(segment, reference, test_words),
        'токенов на слово': np.mean([len(segment(w)) for w in test_words]),
        'обучение, с': train_seconds[name],
        'кодирование, с': float(np.median(times))})
display(pd.DataFrame(comparison))
# BPE здесь байтовый, а в SentencePiece — символьный; границы могут отличаться.
# У Unigram также отличается критерий прунинга и обработка пробелов.

# В проверенном коротком прогоне совпало около 73% разбиений BPE и 40% Unigram.
# Совпадение границ не измеряет качество будущей языковой модели.
# В таблице слова уникальны, поэтому «токенов на слово» здесь отличается
# от fertility всего текста в следующем задании. Кэш BPE очищается перед
# замером, чтобы время не сводилось к поиску уже готовых разбиений.


## 4. Размер словаря, язык и степень сжатия

Для словарей 1k, 8k, 32k (в `FAST_DEV_RUN`: 300, 1000, 2000) обучите BPE на английском и русском корпусах и измерьте на отложенной части:
- **fertility** — среднее число токенов на слово;
- **сжатие** — байтов UTF-8 на токен;
- **долю редких токенов** — долю токенов словаря, встретившихся в отложенной части менее 5 раз.

Обучающая и отложенная части не должны пересекаться.

In [ ]:
VOCABS = [300, 1000, 2000] if FAST_DEV_RUN else [1000, 8000, 32000]

def tokenizer_stats(tok, text, vocab_size):
    ids = tok.encode(text)
    counts = Counter(ids)
    # Fertility считается для всего текстового потока, включая разделители.
    # Размер словаря берём фактический: небольшой корпус может исчерпать пары.
    # В rare_share входят и неиспользованные токены: их частота равна нулю.
    return {'fertility': len(ids) / max(1, len(text.split())),
            'bytes_per_token': len(text.encode('utf-8')) / max(1, len(ids)),
            'rare_share': sum(counts[i] < 5 for i in tok.vocab) / vocab_size}

results = {}
for lang, text in [("en", en_text), ("ru", ru_text)]:
    cut = int(0.8 * len(text)); train_t, test_t = text[:cut], text[cut:]
    for V in VOCABS:
        tok = ByteBPE().train(train_t, V)
        results[(lang, V)] = tokenizer_stats(tok, test_t, len(tok.vocab))
        print(lang, V, {k: round(v, 3) for k, v in results[(lang, V)].items()})

# В коротком прогоне рост словаря уменьшил fertility и увеличил число байтов на токен.
# При словаре 1000 русский текст дал 2.778 токена на слово, английский — 2.505.
# Кириллица занимает больше байтов UTF-8; словоизменение тоже влияет на
# fertility, но итог зависит от обучающего корпуса, а не только от языка.
# У словаря 2000 доля редких токенов оказалась выше, чем у 1000:
# часть добавленных подслов почти не встречается в отложенном тексте.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 3.5))
for ax, key, title in zip(axes, ["fertility", "bytes_per_token", "rare_share"],
                          ["Токенов на слово", "Байтов на токен", "Доля редких токенов"]):
    for lang in ("en", "ru"):
        ax.plot(VOCABS, [results[(lang, V)][key] for V in VOCABS], "o-", label=lang)
    ax.set_xscale("log"); ax.set_xlabel("размер словаря"); ax.set_title(title); ax.legend()
plt.tight_layout(); plt.savefig(ARTIFACTS / "lab01_vocab_curves.png", dpi=120); plt.show()

## 5. Токенизация чисел и кода

Посмотрите, как токенизатор разбивает числа разной длины и фрагменты кода. Найдите примеры, где одно и то же число разбивается по-разному в зависимости от контекста (с пробелом и без), и объясните, почему это мешает модели выполнять арифметику.

In [ ]:
examples = ["12345 + 67890 = 80235", "x12345", " 1000000", "3.14159", "def f(x):\n    return x**2"]
for e in examples:
    print(repr(e), "->", [bpe_en.vocab[i].decode("utf-8", "replace") for i in bpe_en.encode(e)])

# В коротком прогоне цифры остались отдельными токенами. Ведущий пробел
# тоже выводится отдельно, поэтому эти примеры не показывают смену границ
# внутри числа при смене контекста. Правило предтокенизации разрешает группы
# до трёх цифр, но на этом литературном тексте слияния цифр не выучились.
# Если цифры объединены в группы разной длины, токены не совпадают с разрядами:
# перенос при сложении может затрагивать часть одного токена и соседний токен.
# В примере кода return разбит на « re», «t», «urn». Частое слово языка
# программирования не стало одним токеном при обучении на литературном корпусе.


## Вопросы

1. Почему BPE-обучение можно рассматривать как жадный алгоритм сжатия? Что он оптимизирует на каждом шаге и почему результат не оптимален глобально?

   На каждом шаге BPE заменяет частую пару одним токеном и сокращает последовательность. Выбор жадный: он учитывает текущие частоты, но не перебирает будущие слияния. Другой порядок слияний мог бы дать лучшее сжатие всего корпуса.
2. Почему при одинаковом размере словаря русский текст даёт больше токенов на слово, чем английский? Как это влияет на стоимость инференса и эффективную длину контекста?

   Буквы русского алфавита занимают в UTF-8 два байта, ASCII-латиница — один. Разные формы одного русского слова тоже требуют места в словаре. Поэтому для русского текста может понадобиться больше токенов, хотя результат зависит от корпуса и словаря. Больше токенов означает больше вычислений и меньше исходного текста в фиксированном контекстном окне.
3. Чем Unigram LM лучше BPE с вероятностной точки зрения? Что даёт возможность сэмплировать сегментации (subword regularization)?

   Unigram задаёт вероятности разных разбиений слова и при обучении учитывает все допустимые пути. Витерби выбирает самый вероятный путь. Если при обучении языковой модели сэмплировать разные разбиения, она реже привязывается к одному набору подслов: это и есть subword regularization.
4. Почему perplexity моделей с разными токенизаторами нельзя сравнивать напрямую и как перейти к bits-per-byte?

   PPL усредняет ошибку на токен, а токены разных моделей могут содержать разное количество текста. PPL разных токенизаций несопоставима: BPB = NLL / (число байтов * ln(2)). Здесь NLL — суммарная отрицательная логарифмическая вероятность одного и того же тестового текста в натуральных логарифмах. Одних метрик токенизатора для расчёта BPB недостаточно: нужен loss языковой модели.
5. Предложите способ токенизации чисел, облегчающий арифметику, и обоснуйте его.

   Для арифметики можно запретить слияния цифр: границы будут совпадать с разрядами, хотя длина последовательности увеличится. Модели будет проще сопоставлять цифры при сложении и учитывать перенос. Само правило токенизации не обучает модель выполнять арифметические действия.
